# 09 Hugging Face ViT Transfer Learning

## 1. Setup

This notebook fine-tunes a pretrained Hugging Face image-classification
transformer on CIFAR-100 fine labels. It is fully standalone: no local
package imports, no `%%writefile`, no shell git operations.

Run A is a frozen-backbone head baseline. Run B is a partial fine-tune.
Run C (optional) is LoRA via `peft`. LoRA is skipped gracefully if `peft`
fails to install or to wrap the chosen backbone.

[![Open 09 in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fgram-devAI/deepl-cifar100-image-analysis/blob/main/notebooks/09_hf_vit_transfer_learning.ipynb)

In [ ]:
%pip install -q --upgrade pip
%pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib

In [ ]:
RUN_LORA = False  # set to True to attempt LoRA Run C
PEFT_AVAILABLE = False

If you set `RUN_LORA = True`, run this dependency cell first:

In [ ]:
# Guarded equivalent of `%pip install -q peft`.
# Using run_line_magic keeps the cell valid Python when RUN_LORA is False.
if RUN_LORA:
    get_ipython().run_line_magic("pip", "install -q peft")

In [ ]:
if RUN_LORA:
    try:
        import peft  # noqa: F401
        PEFT_AVAILABLE = True
    except Exception as exc:
        print(f"PEFT import failed; LoRA Run C will be skipped: {exc}")
        PEFT_AVAILABLE = False

In [ ]:
import json
import os
import random
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

FAST_DEV_RUN = True       # small subsets, 1 epoch — used for smoke tests
RUN_TRAINING = True       # set False to only inspect data/model
MODEL_NAME = "google/vit-base-patch16-224-in21k"

OUTPUT_ROOT = Path("/content/hf_vit_transfer") if Path("/content").exists() else Path("results/hf_vit_transfer")
RUN_A_DIR = OUTPUT_ROOT / "run_a_frozen_head"
RUN_B_DIR = OUTPUT_ROOT / "run_b_partial_finetune"
RUN_C_DIR = OUTPUT_ROOT / "run_c_lora"

def ensure_dirs() -> None:
    for d in (RUN_A_DIR, RUN_B_DIR, RUN_C_DIR):
        d.mkdir(parents=True, exist_ok=True)

ensure_dirs()
print(f"output root: {OUTPUT_ROOT}")

## 2. Load CIFAR-100 (fine labels, 100 classes)

We load `uoft-cs/cifar100` from Hugging Face Datasets. Validation is a
stratified 0.1 split off the train set. When `FAST_DEV_RUN` is True we
subset to 2,000 train, 500 val, and 500 test rows for quick smoke runs.

In [ ]:
from datasets import load_dataset, ClassLabel

raw = load_dataset("uoft-cs/cifar100")
print(raw)

# Verify fine-label column and capture names.
label_col = "fine_label"
assert label_col in raw["train"].column_names, raw["train"].column_names
NUM_LABELS = raw["train"].features[label_col].num_classes
LABEL_NAMES = raw["train"].features[label_col].names
assert NUM_LABELS == 100, NUM_LABELS

# Stratified split for validation.
split = raw["train"].train_test_split(
    test_size=0.1, stratify_by_column=label_col, seed=SEED
)
raw_train = split["train"]
raw_val = split["test"]
raw_test = raw["test"]

if FAST_DEV_RUN:
    raw_train = raw_train.shuffle(seed=SEED).select(range(2_000))
    raw_val = raw_val.shuffle(seed=SEED).select(range(500))
    raw_test = raw_test.shuffle(seed=SEED).select(range(500))

print(f"train: {len(raw_train)} | val: {len(raw_val)} | test: {len(raw_test)}")
print(f"num_labels: {NUM_LABELS}; sample names: {LABEL_NAMES[:5]}...")

## 3. Why ViT Uses Unmasked Self-Attention for Image Classification

A ViT splits each image into fixed-size patches (e.g. 16×16). Each patch
is linearly embedded into a token, a positional embedding is added, and
the token sequence is fed through a stack of transformer encoder blocks.

For **image classification** the whole image is available at once, so
every patch token is allowed to attend to every other patch token. This
is *bidirectional / unmasked self-attention*. There is no causal order to
enforce, so masking would only throw away useful global context.

*Masked* (causal) self-attention is mainly used in **autoregressive
sequence prediction** — e.g. next-token language models — where the
model must not see future tokens during training. That setup is out of
scope for this image-classification branch and is not implemented here.

In [ ]:
import math
import matplotlib.pyplot as plt
from transformers import AutoConfig

# Inspect the chosen pretrained model's transformer configuration.
cfg = AutoConfig.from_pretrained(MODEL_NAME)
print(f"model: {MODEL_NAME}")
print(f"  hidden_size:        {cfg.hidden_size}")
print(f"  num_hidden_layers:  {cfg.num_hidden_layers}")
print(f"  num_attention_heads:{cfg.num_attention_heads}")
print(f"  patch_size:         {getattr(cfg, 'patch_size', 'n/a')}")
print(f"  image_size:         {getattr(cfg, 'image_size', 'n/a')}")

# Show one CIFAR-100 image split into a conceptual 4x4 patch grid.
sample = raw_train[0]["img"]
fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(sample)
grid = 4
w, h = sample.size
for i in range(1, grid):
    ax.axvline(i * w / grid, color="white", linewidth=0.8)
    ax.axhline(i * h / grid, color="white", linewidth=0.8)
ax.set_title("conceptual patch grid")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 4. Preprocessing — Image Processor and Transforms

We use the model's bundled `AutoImageProcessor` to resize, normalize, and
tensor-convert each PIL image. Training adds a light random horizontal
flip; validation/test use only the processor's deterministic transform.
Labels are surfaced under the key `labels` to satisfy the HF `Trainer`.

In [ ]:
from transformers import AutoImageProcessor
from torchvision.transforms import (
    Compose, Normalize, RandomHorizontalFlip, Resize, ToTensor,
)

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
image_mean = processor.image_mean
image_std = processor.image_std
image_size = processor.size.get("height", 224)

train_tf = Compose([
    Resize((image_size, image_size)),
    RandomHorizontalFlip(p=0.5),
    ToTensor(),
    Normalize(mean=image_mean, std=image_std),
])
eval_tf = Compose([
    Resize((image_size, image_size)),
    ToTensor(),
    Normalize(mean=image_mean, std=image_std),
])

def _apply(batch, tf):
    batch["pixel_values"] = [tf(img.convert("RGB")) for img in batch["img"]]
    batch["labels"] = batch["fine_label"]
    return batch

train_ds = raw_train.with_transform(lambda b: _apply(b, train_tf))
val_ds = raw_val.with_transform(lambda b: _apply(b, eval_tf))
test_ds = raw_test.with_transform(lambda b: _apply(b, eval_tf))

def collate_fn(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "labels": torch.tensor([b["labels"] for b in batch], dtype=torch.long),
    }

# Sanity check one batch shape.
sample_batch = collate_fn([train_ds[i] for i in range(4)])
print({k: tuple(v.shape) for k, v in sample_batch.items()})